# Temporal Anchors

Every temporal offset in the project needs an anchor. This notebook asks how
defensibly that anchor can be recovered when SkyPortal does not provide a
physical trigger time, `t0`.

It starts from NB01's structural source index and uses the same frozen
**2026-07-20** population. Each section follows **question, measurement,
decision**.


In [1]:
from pathlib import Path
import csv
import json
import math
import re
import sys
import warnings

import pandas as pd

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

SOURCE_INDEX_PATH = ROOT / "notebooks" / "evidence" / "01_source_index.csv"
RAW_ROOT = ROOT / "data" / "raw" / "skyportal" / "inventory"
CAPTURE_STAMP = "20260720"
PROFILES = ["grandma_base", "gcn", "ep", "grb"]
STRUCTURAL_COLUMNS = [
    "source_id", "profiles", "name_pattern_class", "has_t0", "t0",
    "n_redshift_versions", "n_summary_versions", "n_classifications",
    "has_alias", "has_tns_name", "comment_exists", "photometry_exists",
    "spectrum_exists", "created_at", "modified", "ra", "dec", "redshift",
]
ANCHOR_COLUMNS = [
    "anchor_type", "t0_from_id", "id_timestamp", "delta_id_created_s",
    "t0_source", "t0_uncertainty_hours", "tier_status",
]

with SOURCE_INDEX_PATH.open("r", encoding="utf-8", newline="") as handle:
    reader = csv.DictReader(handle)
    input_columns = list(reader.fieldnames or [])
    input_rows = list(reader)

missing_columns = [column for column in STRUCTURAL_COLUMNS if column not in input_columns]
if missing_columns:
    raise ValueError(f"NB01 structural columns are missing: {missing_columns}")

structural_rows_raw = [
    {column: row[column] for column in STRUCTURAL_COLUMNS}
    for row in input_rows
]
source_index = pd.read_csv(SOURCE_INDEX_PATH, usecols=STRUCTURAL_COLUMNS)
current_t0 = source_index.copy()
current_t0["has_t0"] = (
    current_t0["has_t0"].astype(str).str.lower().eq("true")
)
for flag in ["comment_exists", "photometry_exists", "spectrum_exists"]:
    current_t0[flag] = current_t0[flag].astype(str).str.lower().eq("true")
current_t0["n_classifications"] = pd.to_numeric(
    current_t0["n_classifications"], errors="coerce"
).fillna(0).astype(int)
current_t0["has_classifications"] = current_t0["n_classifications"].gt(0)


print(f"Python: {sys.executable}")
print(f"Input rows: {len(current_t0)}")


Python: /home/meneses/project_astronomical/MAFORAI/.venv/bin/python
Input rows: 800


## 1. The problem

**QUESTION.** How many of the 800 sources lack a real `t0`, and how many of
those sources already carry scientific data that makes reconstruction useful?


In [2]:
flag_rows = []
for flag in [
    "comment_exists", "photometry_exists", "spectrum_exists",
    "has_classifications",
]:
    flag_rows.append(
        {
            "data_bearing_flag": flag,
            "without_t0": int(current_t0.loc[~current_t0["has_t0"], flag].sum()),
            "with_t0": int(current_t0.loc[current_t0["has_t0"], flag].sum()),
            "all_sources": int(current_t0[flag].sum()),
        }
    )

name_t0 = pd.crosstab(
    current_t0["name_pattern_class"], current_t0["has_t0"]
).rename(columns={False: "without_t0", True: "with_t0"})
for column in ["without_t0", "with_t0"]:
    if column not in name_t0:
        name_t0[column] = 0
name_t0["all_sources"] = name_t0["without_t0"] + name_t0["with_t0"]
name_t0["pct_with_t0"] = 100.0 * name_t0["with_t0"] / name_t0["all_sources"]

missing_t0 = current_t0.loc[~current_t0["has_t0"]]
missing_t0_has_data = missing_t0[
    ["comment_exists", "photometry_exists", "spectrum_exists", "has_classifications"]
].any(axis=1)

print(pd.DataFrame(flag_rows).to_string(index=False))
print(name_t0.reset_index().to_string(
    index=False, formatters={"pct_with_t0": lambda value: f"{value:.2f}%"}
))
print(f"Sources with t0: {int(current_t0['has_t0'].sum())}/{len(current_t0)}")
print(f"Sources without t0: {len(missing_t0)}/{len(current_t0)}")
print(
    "Sources without t0 carrying at least one data-bearing flag: "
    f"{int(missing_t0_has_data.sum())}/{len(missing_t0)}"
)


  data_bearing_flag  without_t0  with_t0  all_sources
     comment_exists         233      118          351
  photometry_exists         138      102          240
    spectrum_exists           0        1            1
has_classifications         192       59          251
name_pattern_class  without_t0  with_t0  all_sources pct_with_t0
       ep_internal         189        5          194       2.58%
      gcn_internal         102       53          155      34.19%
      grb_internal          74       31          105      29.52%
         grb_named          67       27           94      28.72%
             other         203       11          214       5.14%
          tns_like          13        0           13       0.00%
          ztf_like          24        1           25       4.00%
Sources with t0: 128/800
Sources without t0: 672/800
Sources without t0 carrying at least one data-bearing flag: 296/672


**FINDING.** Only **128 of 800** sources have a real `t0`. Of the **672**
without one, **296** already carry at least one data-bearing flag. Temporal
reconstruction therefore concerns a substantial active subset, not an empty
tail of the inventory.


## 2. Two kinds of anchor

**QUESTION.** Which sources have a physical trigger, and which can only be
anchored by their first detection?


In [3]:
trigger_name_classes = {
    "gcn_internal", "ep_internal", "grb_internal", "grb_named",
}
current_t0["anchor_type"] = "first_detection"
current_t0.loc[
    current_t0["name_pattern_class"].isin(trigger_name_classes),
    "anchor_type",
] = "trigger"

anchor_coverage = (
    current_t0.groupby("anchor_type")["has_t0"]
    .agg(sources="size", sources_with_t0="sum")
    .reset_index()
)
anchor_coverage["sources_without_t0"] = (
    anchor_coverage["sources"] - anchor_coverage["sources_with_t0"]
)
anchor_coverage["pct_with_t0"] = (
    100.0 * anchor_coverage["sources_with_t0"] / anchor_coverage["sources"]
)
print(anchor_coverage.to_string(
    index=False, formatters={"pct_with_t0": lambda value: f"{value:.2f}%"}
))


    anchor_type  sources  sources_with_t0  sources_without_t0 pct_with_t0
first_detection      252               12                 240       4.76%
        trigger      548              116                 432      21.17%


**DECISION.** Treat the **548 trigger anchors** and **252 first-detection
anchors** as different semantics. A trigger is a measured physical instant. A
first detection is an upper bound on explosion time. Consequently, $\Delta t$
means time since trigger for the first group and time since first detection for
the second; the quantities are not interchangeable.


## 3. Does the internal ID encode the trigger time?

**QUESTION.** How closely does `PREFIX-YYMMDD_HHMMSS` agree with a real `t0`
where both are available?


In [4]:
id_parts = current_t0["source_id"].str.extract(
    r"^(?P<id_prefix>GCN|EP|GRB)-(?P<id_date>\d{6})_(?P<id_clock>\d{6})$",
    flags=re.IGNORECASE,
)
id_timestamp = pd.to_datetime(
    id_parts["id_date"].fillna("") + id_parts["id_clock"].fillna(""),
    format="%y%m%d%H%M%S", errors="coerce", utc=True,
)
current_t0["id_prefix"] = id_parts["id_prefix"].str.upper()
current_t0["t0_from_id"] = (
    (id_timestamp - pd.Timestamp("1858-11-17", tz="UTC"))
    / pd.Timedelta(days=1)
)

id_pattern_matches = id_parts["id_prefix"].notna()
id_parse_failures = current_t0.loc[
    id_pattern_matches & current_t0["t0_from_id"].isna(), "source_id"
]
id_validation = current_t0.loc[
    current_t0["has_t0"] & current_t0["t0_from_id"].notna(),
    ["source_id", "id_prefix", "t0", "t0_from_id"],
].copy()
id_validation["delta_seconds"] = (
    pd.to_numeric(id_validation["t0_from_id"])
    - pd.to_numeric(id_validation["t0"])
) * 86400.0
id_validation["abs_delta_seconds"] = id_validation["delta_seconds"].abs()

prefix_rows = []
for prefix, group in id_validation.groupby("id_prefix", sort=True):
    prefix_rows.append(
        {
            "prefix": prefix,
            "n": len(group),
            "min_s": group["delta_seconds"].min(),
            "p25_s": group["delta_seconds"].quantile(0.25),
            "median_s": group["delta_seconds"].median(),
            "p75_s": group["delta_seconds"].quantile(0.75),
            "p95_s": group["delta_seconds"].quantile(0.95),
            "max_s": group["delta_seconds"].max(),
            "abs_over_5_min": int((group["abs_delta_seconds"] > 300).sum()),
            "abs_over_1_h": int((group["abs_delta_seconds"] > 3600).sum()),
            "abs_over_6_h": int((group["abs_delta_seconds"] > 21600).sum()),
            "assessment": "indicative only" if len(group) < 10 else "measured",
        }
    )
prefix_id_offsets = pd.DataFrame(prefix_rows)
ep_control = id_validation.loc[
    id_validation["source_id"].eq("EP-260623_025405")
]

print(f"Internal IDs matching the pattern: {int(id_pattern_matches.sum())}")
print(f"Parsed successfully: {int(current_t0['t0_from_id'].notna().sum())}")
print(f"Parse failures: {len(id_parse_failures)}")
print(f"Validation set size: {len(id_validation)}")
print(prefix_id_offsets.to_string(
    index=False, float_format=lambda value: f"{value:.3f}"
))
print("EP-260623_025405 control:")
print(ep_control.to_string(index=False, float_format=lambda value: f"{value:.8f}"))


Internal IDs matching the pattern: 454
Parsed successfully: 454
Parse failures: 0
Validation set size: 89
prefix  n     min_s   p25_s  median_s  p75_s   p95_s       max_s  abs_over_5_min  abs_over_1_h  abs_over_6_h      assessment
    EP  5    -0.000   0.000    26.000 98.000 794.944     969.180               1             0             0 indicative only
   GCN 53 -1310.000 -20.000    -5.000  0.000  14.920    1221.374               3             0             0        measured
   GRB 31    -0.458  -0.000     0.000  1.000  13.250 1034457.000               1             1             1        measured
EP-260623_025405 control:
       source_id id_prefix             t0     t0_from_id  delta_seconds  abs_delta_seconds
EP-260623_025405        EP 61214.11975694 61214.12089120    98.00038312        98.00038312


**FINDING.** All **454** matching IDs parse successfully, and **89** sources
provide direct validation against a real `t0`. GCN has 53 controls, EP only 5,
and GRB includes one measured **287.349 h** failure. The
`EP-260623_025405` identifier is approximately **98 s** later than its real
`t0`.


## 4. What does the ID actually encode?

**QUESTION.** Is the internal timestamp a physical-event timestamp, a copy of
`created_at`, or a record-creation stamp that normally follows the trigger
closely?


### 4.1. ID timestamp versus `created_at`

**QUESTION.** Across all 454 parseable IDs, how far apart are the ID timestamp
and source-row creation time?


In [5]:
created_at_timestamp = pd.to_datetime(
    current_t0["created_at"], errors="coerce", utc=True
)
current_t0["id_timestamp"] = id_timestamp
current_t0["delta_id_created_s"] = (
    created_at_timestamp - id_timestamp
).dt.total_seconds()

parseable_id_mask = current_t0["id_timestamp"].notna()
delta_all = current_t0.loc[parseable_id_mask, "delta_id_created_s"]
overall_delta = pd.DataFrame(
    [{
        "n": len(delta_all),
        "min_s": delta_all.min(),
        "p05_s": delta_all.quantile(0.05),
        "p25_s": delta_all.quantile(0.25),
        "median_s": delta_all.median(),
        "p75_s": delta_all.quantile(0.75),
        "p95_s": delta_all.quantile(0.95),
        "max_s": delta_all.max(),
        "abs_over_5_min": int((delta_all.abs() > 300).sum()),
        "abs_over_1_h": int((delta_all.abs() > 3600).sum()),
        "abs_over_1_day": int((delta_all.abs() > 86400).sum()),
    }]
)
print(overall_delta.to_string(
    index=False, float_format=lambda value: f"{value:.3f}"
))


  n      min_s  p05_s   p25_s  median_s    p75_s     p95_s      max_s  abs_over_5_min  abs_over_1_h  abs_over_1_day
454 -56635.904 41.866 241.434   488.450 1521.501 42907.111 705466.738             315            65              12


**FINDING.** The ID and `created_at` are usually close, which is consistent
with both being generated during record ingestion. This comparison alone does
not prove that either one is the physical trigger time.


### 4.2. Prefix tails

**QUESTION.** Do GCN and EP have comparable ID-to-creation tails, not merely
comparable medians?


In [6]:
sub = current_t0.loc[parseable_id_mask].copy()
sub["abs_delta"] = sub["delta_id_created_s"].abs()

prefix_delta_rows = []
for prefix in sorted(sub["id_prefix"].dropna().unique()):
    group = sub.loc[sub["id_prefix"] == prefix, "delta_id_created_s"]
    abs_group = group.abs()
    prefix_delta_rows.append(
        {
            "prefix": prefix,
            "n": len(group),
            "min_s": group.min(),
            "p05_s": group.quantile(0.05),
            "p25_s": group.quantile(0.25),
            "median_s": group.median(),
            "p75_s": group.quantile(0.75),
            "p95_s": group.quantile(0.95),
            "max_s": group.max(),
            "abs_over_5_min": int((abs_group > 300).sum()),
            "abs_over_1_h": int((abs_group > 3600).sum()),
            "abs_over_1_day": int((abs_group > 86400).sum()),
        }
    )
prefix_delta = pd.DataFrame(prefix_delta_rows)

def fisher_exact_two_sided(a, b, c, d):
    """Return a two-sided Fisher exact p-value without scipy."""
    n, row1, row2, col1 = a + b + c + d, a + b, c + d, a + c
    def hyper_p(x):
        return (
            math.comb(row1, x) * math.comb(row2, col1 - x)
            / math.comb(n, col1)
        )
    observed_p = hyper_p(a)
    lo, hi = max(0, col1 - row2), min(row1, col1)
    return sum(
        hyper_p(x) for x in range(lo, hi + 1)
        if hyper_p(x) <= observed_p * 1.0000001
    )

ep_row = prefix_delta.loc[prefix_delta["prefix"] == "EP"].iloc[0]
gcn_row = prefix_delta.loc[prefix_delta["prefix"] == "GCN"].iloc[0]
tail_rate_p_value = fisher_exact_two_sided(
    int(ep_row["abs_over_1_day"]),
    int(ep_row["n"] - ep_row["abs_over_1_day"]),
    int(gcn_row["abs_over_1_day"]),
    int(gcn_row["n"] - gcn_row["abs_over_1_day"]),
)
tail_p95_ratio = ep_row["p95_s"] / gcn_row["p95_s"]

print(prefix_delta.to_string(
    index=False, float_format=lambda value: f"{value:.3f}"
))
print(
    f"EP over-1-day rate: {int(ep_row['abs_over_1_day'])}/{int(ep_row['n'])} "
    f"({100 * ep_row['abs_over_1_day'] / ep_row['n']:.2f}%); "
    f"GCN: {int(gcn_row['abs_over_1_day'])}/{int(gcn_row['n'])} "
    f"({100 * gcn_row['abs_over_1_day'] / gcn_row['n']:.2f}%)"
)
print(f"EP/GCN p95 ratio: {tail_p95_ratio:.1f}x")
print(f"Two-sided Fisher exact p-value: {tail_rate_p_value:.4f}")


prefix   n      min_s   p05_s   p25_s  median_s    p75_s     p95_s      max_s  abs_over_5_min  abs_over_1_h  abs_over_1_day
    EP 194 -56635.904 264.225 347.624   679.580 3529.435 62988.860 610366.094             171            50               6
   GCN 155     35.709  73.140 413.299   845.206 1358.310  3500.132 478207.473             120             8               1
   GRB 105     31.540  35.734  45.299    85.965  233.556  7296.685 705466.738              24             7               5
EP over-1-day rate: 6/194 (3.09%); GCN: 1/155 (0.65%)
EP/GCN p95 ratio: 18.0x
Two-sided Fisher exact p-value: 0.1377


**FINDING.** GCN has an ID-to-creation p95 near **1.0 h**, while EP's is
**17.5 h**, an approximately **18x** difference. Their over-one-day rates are
0.65% and 3.09%. Fisher's exact $p=0.14$ does not resolve that difference, but
it also does not establish equal tail risk.


### 4.3. Three-way validation

**QUESTION.** On the 89 direct controls, is the ID timestamp closer to real
`t0` than `created_at` is?


In [7]:
t0_mjd = pd.to_numeric(current_t0["t0"], errors="coerce")
with warnings.catch_warnings():
    warnings.simplefilter("ignore", RuntimeWarning)
    t0_timestamp = (
        pd.Timestamp("1858-11-17", tz="UTC")
        + pd.to_timedelta(t0_mjd, unit="D")
    )
three_way = current_t0.loc[
    current_t0["has_t0"] & current_t0["id_timestamp"].notna()
].copy()
three_way["t0_timestamp"] = t0_timestamp.loc[three_way.index]
three_way["id_minus_t0_s"] = (
    three_way["id_timestamp"] - three_way["t0_timestamp"]
).dt.total_seconds()
three_way["created_minus_t0_s"] = (
    created_at_timestamp.loc[three_way.index] - three_way["t0_timestamp"]
).dt.total_seconds()
three_way["created_minus_id_s"] = three_way["delta_id_created_s"]

three_way_summary_rows = []
for label, group in [
    ("all", three_way), *list(three_way.groupby("id_prefix", sort=True))
]:
    three_way_summary_rows.append(
        {
            "prefix": label,
            "n": len(group),
            "median_abs_id_minus_t0_s": group["id_minus_t0_s"].abs().median(),
            "median_abs_created_minus_t0_s": group["created_minus_t0_s"].abs().median(),
            "median_created_minus_id_s": group["created_minus_id_s"].median(),
        }
    )
three_way_summary = pd.DataFrame(three_way_summary_rows)
all_row = three_way_summary.loc[three_way_summary["prefix"] == "all"].iloc[0]
tighter_ratio = (
    all_row["median_abs_created_minus_t0_s"]
    / all_row["median_abs_id_minus_t0_s"]
)

print(three_way_summary.to_string(
    index=False, float_format=lambda value: f"{value:.3f}"
))
print(
    f"Median absolute offset: ID {all_row['median_abs_id_minus_t0_s']:.1f} s; "
    f"created_at {all_row['median_abs_created_minus_t0_s']:.1f} s; "
    f"ratio {tighter_ratio:.1f}x"
)


prefix  n  median_abs_id_minus_t0_s  median_abs_created_minus_t0_s  median_created_minus_id_s
   all 89                     2.000                        110.661                    119.177
    EP  5                    26.000                       2479.218                   1510.038
   GCN 53                     7.000                        544.813                    461.780
   GRB 31                     0.458                         68.088                     62.767
Median absolute offset: ID 2.0 s; created_at 110.7 s; ratio 55.3x


**FINDING.** The ID timestamp is not a copy of `created_at`: its median
absolute offset from real `t0` is approximately **2 s**, versus **111 s** for
`created_at`, making the ID about **55x tighter** on the direct controls.


### 4.4. The GRB outlier

**QUESTION.** What does the measured 287-hour GRB failure reveal about the
mechanism?


In [8]:
grb_outlier = current_t0.loc[
    current_t0["source_id"] == "GRB-250329_041752"
].copy()
grb_outlier["t0_timestamp"] = t0_timestamp.loc[grb_outlier.index]
outlier_id_minus_t0 = (
    grb_outlier["id_timestamp"].iloc[0]
    - grb_outlier["t0_timestamp"].iloc[0]
).total_seconds()
outlier_created_minus_t0 = (
    pd.to_datetime(grb_outlier["created_at"].iloc[0], utc=True)
    - grb_outlier["t0_timestamp"].iloc[0]
).total_seconds()
outlier_created_minus_id = float(
    grb_outlier["delta_id_created_s"].iloc[0]
)

p95_by_prefix = sub.groupby("id_prefix")["abs_delta"].quantile(0.95)
sub["p95_for_own_prefix"] = sub["id_prefix"].map(p95_by_prefix)
exceeders = sub.loc[sub["abs_delta"] > sub["p95_for_own_prefix"]]

print(grb_outlier[
    ["source_id", "t0_timestamp", "id_timestamp", "created_at"]
].to_string(index=False))
print(f"ID minus t0: {outlier_id_minus_t0 / 3600:.3f} h")
print(f"created_at minus t0: {outlier_created_minus_t0 / 3600:.3f} h")
print(f"created_at minus ID: {outlier_created_minus_id:.1f} s")
print(f"Sources above their prefix p95: {len(exceeders)}")
print(exceeders["id_prefix"].value_counts().rename_axis("prefix").to_string())


        source_id                        t0_timestamp              id_timestamp                 created_at
GRB-250329_041752 2025-03-17 04:56:55.000031775+00:00 2025-03-29 04:17:52+00:00 2025-03-29T04:18:54.766979
ID minus t0: 287.349 h
created_at minus t0: 287.367 h
created_at minus ID: 62.8 s
Sources above their prefix p95: 24
prefix
EP     10
GCN     8
GRB     6


**FINDING.** For `GRB-250329_041752`, both the ID and `created_at` are about
**287.349 h** late, yet they remain only about **63 s** apart. The entire record
was created late; the ID did not independently preserve the physical trigger.


### 4.5. Can a late record be detected internally?

**QUESTION.** Can `source_id` and `created_at` alone reveal that both are late?


In [9]:
late_record_check = pd.DataFrame(
    [{
        "source_id": "GRB-250329_041752",
        "id_minus_t0_h": outlier_id_minus_t0 / 3600.0,
        "created_minus_t0_h": outlier_created_minus_t0 / 3600.0,
        "created_minus_id_s": outlier_created_minus_id,
        "detectable_without_external_time": False,
    }]
)
print(late_record_check.to_string(
    index=False, float_format=lambda value: f"{value:.3f}"
))


        source_id  id_minus_t0_h  created_minus_t0_h  created_minus_id_s  detectable_without_external_time
GRB-250329_041752        287.349             287.367              62.767                             False


**DECISION.** No. A late record is invisible when its ID timestamp and
`created_at` drift together. Detection needs an independent physical signal:
first photometry or a GCN `TRIGGER_TIME`. This limitation keeps EP provisional
despite its tight median behavior.


## 5. `created_at` as a floor

**QUESTION.** How far after real `t0` can source-row creation occur, and can it
serve as a physical anchor?


In [10]:
created_at_mjd = (
    (created_at_timestamp - pd.Timestamp("1858-11-17", tz="UTC"))
    / pd.Timedelta(days=1)
)
created_validation = current_t0.loc[
    current_t0["has_t0"] & created_at_timestamp.notna(),
    ["source_id", "anchor_type", "t0"],
].copy()
created_validation["delta_hours"] = (
    created_at_mjd.loc[created_validation.index]
    - pd.to_numeric(created_validation["t0"])
) * 24.0

created_rows = []
for anchor_label, group in [
    ("all", created_validation),
    *list(created_validation.groupby("anchor_type", sort=True)),
]:
    created_rows.append(
        {
            "anchor_type": anchor_label,
            "n": len(group),
            "min_h": group["delta_hours"].min(),
            "median_h": group["delta_hours"].median(),
            "p90_h": group["delta_hours"].quantile(0.90),
            "max_h": group["delta_hours"].max(),
        }
    )
created_at_offsets = pd.DataFrame(created_rows)
print(created_at_offsets.to_string(
    index=False, float_format=lambda value: f"{value:.3f}"
))


    anchor_type   n  min_h  median_h   p90_h     max_h
            all 128  0.005     0.171  24.653 17572.145
first_detection  12  1.091    16.036 178.095 17572.145
        trigger 116  0.005     0.116  10.294   349.538


**DECISION.** `created_at` is a knowledge-time floor, not a physical anchor.
Its maximum observed lag is **349.538 h** for trigger sources and
**17,572.145 h** for first-detection sources. It may order a dossier but must
not support phase matching.


## 6. The measured ladder

**QUESTION.** Which fallback tiers are supported for phase matching, which are
provisional, and which provide dossier context only?

**PHASE MATCHING** accepts temporal phase comparisons. **PROVISIONAL** retains
an anchor but excludes it from phase matching pending named validation.
**DOSSIER ONLY** provides ordering or context, not a physical phase anchor.


In [11]:
missing_real_t0 = ~current_t0["has_t0"]
id_fallback = missing_real_t0 & current_t0["t0_from_id"].notna()
gcn_id_fallback = id_fallback & current_t0["id_prefix"].eq("GCN")
ep_id_fallback = id_fallback & current_t0["id_prefix"].eq("EP")
grb_id_fallback = id_fallback & current_t0["id_prefix"].eq("GRB")
pending_gcn_trigger_time = (
    missing_real_t0
    & current_t0["anchor_type"].eq("trigger")
    & current_t0["t0_from_id"].isna()
)
pending_first_detection = (
    missing_real_t0 & current_t0["anchor_type"].eq("first_detection")
)
created_at_fallback = missing_real_t0 & ~id_fallback

id_p95_abs_hours = (
    id_validation.groupby("id_prefix")["abs_delta_seconds"].quantile(0.95)
    / 3600.0
)
id_max_abs_hours = (
    id_validation.groupby("id_prefix")["abs_delta_seconds"].max()
    / 3600.0
)
created_max_hours = created_validation.groupby("anchor_type")["delta_hours"].max()

ladder = pd.DataFrame(
    [
        {
            "tier": "SkyPortal t0",
            "rule": "Use the populated listing t0",
            "n_sources": int(current_t0["has_t0"].sum()),
            "measured_uncertainty": "direct field; 0 h assigned",
            "validated_against": "listing value",
            "status": "PHASE MATCHING",
        },
        {
            "tier": "GCN internal ID",
            "rule": "Parse GCN-YYMMDD_HHMMSS",
            "n_sources": int(gcn_id_fallback.sum()),
            "measured_uncertainty": (
                f"p95 abs {id_p95_abs_hours['GCN']:.3f} h; "
                f"max abs {id_max_abs_hours['GCN']:.3f} h"
            ),
            "validated_against": (
                f"{int((id_validation['id_prefix'] == 'GCN').sum())} real t0 values"
            ),
            "status": "PHASE MATCHING",
        },
        {
            "tier": "EP internal ID",
            "rule": "Parse EP-YYMMDD_HHMMSS",
            "n_sources": int(ep_id_fallback.sum()),
            "measured_uncertainty": (
                f"p95 abs {id_p95_abs_hours['EP']:.3f} h; "
                f"max abs {id_max_abs_hours['EP']:.3f} h"
            ),
            "validated_against": "5 real t0 values; photometry causality pending",
            "status": "PROVISIONAL",
        },
        {
            "tier": "GRB internal ID",
            "rule": "Parse GRB-YYMMDD_HHMMSS",
            "n_sources": int(grb_id_fallback.sum()),
            "measured_uncertainty": (
                f"p95 abs {id_p95_abs_hours['GRB']:.3f} h; "
                f"max abs {id_max_abs_hours['GRB']:.3f} h"
            ),
            "validated_against": (
                f"{int((id_validation['id_prefix'] == 'GRB').sum())} real t0 values"
            ),
            "status": "DOSSIER ONLY; OUTLIER",
        },
        {
            "tier": "GCN TRIGGER_TIME",
            "rule": "Extract trigger time from associated circulars",
            "n_sources": int(pending_gcn_trigger_time.sum()),
            "measured_uncertainty": "PENDING",
            "validated_against": "future GCN span-store step",
            "status": "PENDING",
        },
        {
            "tier": "First detection",
            "rule": "Use earliest validated survey detection",
            "n_sources": int(pending_first_detection.sum()),
            "measured_uncertainty": "PENDING",
            "validated_against": "future full photometry ingestion",
            "status": "PENDING",
        },
        {
            "tier": "SkyPortal created_at",
            "rule": "Use row creation time as knowledge-time floor",
            "n_sources": int(created_at_fallback.sum()),
            "measured_uncertainty": (
                f"trigger max {created_max_hours['trigger']:.3f} h; "
                f"first-detection max {created_max_hours['first_detection']:.3f} h"
            ),
            "validated_against": f"{len(created_validation)} real t0 values",
            "status": "DOSSIER ONLY",
        },
    ]
)


print(ladder.to_string(index=False))


                tier                                           rule  n_sources                                   measured_uncertainty                              validated_against                status
        SkyPortal t0                   Use the populated listing t0        128                             direct field; 0 h assigned                                  listing value        PHASE MATCHING
     GCN internal ID                        Parse GCN-YYMMDD_HHMMSS        102                       p95 abs 0.057 h; max abs 0.364 h                              53 real t0 values        PHASE MATCHING
      EP internal ID                         Parse EP-YYMMDD_HHMMSS        189                       p95 abs 0.221 h; max abs 0.269 h 5 real t0 values; photometry causality pending           PROVISIONAL
     GRB internal ID                        Parse GRB-YYMMDD_HHMMSS         74                     p95 abs 0.004 h; max abs 287.349 h                              31 real t0 values DOSSIER

**FINDING.** The measured evidence supports direct SkyPortal `t0` and GCN IDs
for phase matching, keeps EP provisional, and rejects GRB IDs and `created_at`
as physical phase anchors.

### 6.1. Apply and persist the ladder

**QUESTION.** What status does each source receive, and can the anchor columns
be added without changing any structural value authored by NB01?


In [12]:
current_t0["t0_source"] = "created_at_dossier_only"
current_t0["t0_uncertainty_hours"] = current_t0["anchor_type"].map(
    created_max_hours
)
current_t0.loc[current_t0["has_t0"], "t0_source"] = "skyportal_t0"
current_t0.loc[current_t0["has_t0"], "t0_uncertainty_hours"] = 0.0
for prefix, mask in [
    ("GCN", gcn_id_fallback),
    ("EP", ep_id_fallback),
    ("GRB", grb_id_fallback),
]:
    current_t0.loc[mask, "t0_source"] = f"source_id_timestamp_{prefix.lower()}"
    current_t0.loc[mask, "t0_uncertainty_hours"] = id_max_abs_hours[prefix]

current_t0["tier_status"] = "dossier_only"
current_t0.loc[current_t0["has_t0"], "tier_status"] = "phase_matching"
current_t0.loc[gcn_id_fallback, "tier_status"] = "phase_matching"
current_t0.loc[ep_id_fallback, "tier_status"] = "provisional"

selected_anchor_mjd = pd.to_numeric(current_t0["t0"], errors="coerce")
selected_anchor_mjd = selected_anchor_mjd.where(
    selected_anchor_mjd.notna(), current_t0["t0_from_id"]
)
selected_anchor_mjd = selected_anchor_mjd.where(
    selected_anchor_mjd.notna(), created_at_mjd
)
anchorless_count = int(selected_anchor_mjd.isna().sum())
status_counts = (
    current_t0["tier_status"].value_counts()
    .reindex(["phase_matching", "provisional", "dossier_only"], fill_value=0)
    .rename_axis("tier_status").reset_index(name="sources")
)

def serialize_anchor(value):
    if pd.isna(value):
        return ""
    if isinstance(value, pd.Timestamp):
        return value.isoformat()
    if isinstance(value, float):
        return format(value, ".15g")
    return str(value)

augmented_rows = []
for index, structural_row in enumerate(structural_rows_raw):
    computed = current_t0.iloc[index]
    augmented = dict(structural_row)
    for column in ANCHOR_COLUMNS:
        augmented[column] = serialize_anchor(computed[column])
    augmented_rows.append(augmented)

temporary_path = SOURCE_INDEX_PATH.with_suffix(".csv.part")
with temporary_path.open("w", encoding="utf-8", newline="") as handle:
    writer = csv.DictWriter(
        handle, fieldnames=STRUCTURAL_COLUMNS + ANCHOR_COLUMNS
    )
    writer.writeheader()
    writer.writerows(augmented_rows)
temporary_path.replace(SOURCE_INDEX_PATH)

with SOURCE_INDEX_PATH.open("r", encoding="utf-8", newline="") as handle:
    output_rows = list(csv.DictReader(handle))
structural_rows_after = [
    {column: row[column] for column in STRUCTURAL_COLUMNS}
    for row in output_rows
]
structural_values_preserved = structural_rows_after == structural_rows_raw
if not structural_values_preserved:
    raise AssertionError("NB01 structural values changed while adding anchors")

print("Status totals:")
print(status_counts.to_string(index=False))
print(f"Sources with no anchor at all: {anchorless_count}")
print(f"Structural values preserved exactly: {structural_values_preserved}")
print(f"Output columns: {STRUCTURAL_COLUMNS + ANCHOR_COLUMNS}")


Status totals:
   tier_status  sources
phase_matching      230
   provisional      189
  dossier_only      381
Sources with no anchor at all: 0
Structural values preserved exactly: True
Output columns: ['source_id', 'profiles', 'name_pattern_class', 'has_t0', 't0', 'n_redshift_versions', 'n_summary_versions', 'n_classifications', 'has_alias', 'has_tns_name', 'comment_exists', 'photometry_exists', 'spectrum_exists', 'created_at', 'modified', 'ra', 'dec', 'redshift', 'anchor_type', 't0_from_id', 'id_timestamp', 'delta_id_created_s', 't0_source', 't0_uncertainty_hours', 'tier_status']


**DECISION.** Accept populated SkyPortal `t0` and validated GCN internal IDs
for phase matching. Keep EP IDs provisional pending an external photometry
causality check. Keep GRB IDs and `created_at` dossier-only. The applied ladder
yields **230 phase-matching**, **189 provisional**, and **381 dossier-only**
sources; none is anchorless, but 307 rely only on a knowledge-time floor.


## 7. Decision

**QUESTION.** What temporal population can be used now, and what remains
scientifically unresolved?


In [13]:
july_rows = []
for profile_name in PROFILES:
    run_directories = sorted(
        RAW_ROOT.glob(f"source_inventory_{profile_name}_{CAPTURE_STAMP}_*")
    )
    if len(run_directories) != 1:
        raise RuntimeError(
            f"Expected one frozen run for {profile_name}, found {len(run_directories)}"
        )
    for page_path in sorted(run_directories[0].glob("sources_page_*.json")):
        payload = json.loads(page_path.read_text(encoding="utf-8"))
        july_rows.extend(
            row for row in payload.get("data", {}).get("sources", [])
            if isinstance(row, dict)
        )

phase_matching_n = int((current_t0["tier_status"] == "phase_matching").sum())
provisional_n = int((current_t0["tier_status"] == "provisional").sum())
dossier_only_n = int((current_t0["tier_status"] == "dossier_only").sum())
ep_direct_controls = int((id_validation["id_prefix"] == "EP").sum())
ep_failure_upper_95 = 100.0 * (1.0 - 0.05 ** (1.0 / ep_direct_controls))
n_spectrum_exists = int(current_t0["spectrum_exists"].sum())
n_annotations_nonempty_records = sum(
    record.get("annotations") not in (None, "", [], {})
    for record in july_rows
)
prefix_real_t0 = (
    current_t0.loc[current_t0["id_prefix"].isin(["GCN", "EP", "GRB"])]
    .groupby("id_prefix")["has_t0"]
    .agg(sources="size", with_t0="sum")
)
prefix_real_t0["pct_with_t0"] = (
    100.0 * prefix_real_t0["with_t0"] / prefix_real_t0["sources"]
)

decision_metrics = pd.DataFrame(
    [
        ("phase_matching", phase_matching_n),
        ("provisional", provisional_n),
        ("dossier_only", dossier_only_n),
        ("EP direct controls", ep_direct_controls),
        ("EP 95% failure-rate upper bound (%)", ep_failure_upper_95),
        ("spectrum_exists sources", n_spectrum_exists),
        ("raw records with skyportal annotations", n_annotations_nonempty_records),
    ],
    columns=["metric", "value"],
)
print(decision_metrics.to_string(
    index=False, float_format=lambda value: f"{value:.2f}"
))
print("Real t0 coverage by internal prefix:")
print(prefix_real_t0.to_string(
    formatters={"pct_with_t0": lambda value: f"{value:.2f}%"}
))


                                metric  value
                        phase_matching 230.00
                           provisional 189.00
                          dossier_only 381.00
                    EP direct controls   5.00
   EP 95% failure-rate upper bound (%)  45.07
               spectrum_exists sources   1.00
raw records with skyportal annotations   3.00
Real t0 coverage by internal prefix:
           sources  with_t0 pct_with_t0
id_prefix                              
EP             194        5       2.58%
GCN            155       53      34.19%
GRB            105       31      29.52%


**DECISION.** **230 of 800 sources are phase-matchable, 189 are provisional,
and 381 are dossier-only.**

- Populated SkyPortal `t0` supports 128 sources. Another 102 GCN ID fallbacks
  enter phase matching, supported by 53 controls and a maximum measured error
  of 0.364 h.
- EP remains **PROVISIONAL**. Its p95 tail differs from GCN by about 18x; five
  direct controls with zero failures still permit an approximately 45% upper
  bound on the true failure rate; and internal fields cannot reveal a record
  created late. Full-photometry causality validation is pending.
- GRB IDs remain dossier-only after one of 31 controls failed by 287.349 h.
  The remaining 307 `created_at` fallbacks are also dossier-only knowledge-time
  anchors.
- Real `t0` covers only 2.6% of EP internal sources versus 34.2% of GCN internal
  sources. This weakens any assumption that both prefixes share identical
  ingestion behavior.
- The pending GCN `TRIGGER_TIME` and first-detection tiers remain unresolved.
  `spectrum_exists` is true for 1 of 800 sources, and non-empty SkyPortal
  `annotations` occur in 3 of 982 raw records; both remain residual fact types.


## Decisions summary

The table records only decisions supported by the measurements above.


In [14]:
decisions = pd.DataFrame(
    [
        ("Reconstruct missing anchors", "672 lack t0; 296 are data-bearing", 1),
        ("Separate trigger and first detection", "548 trigger / 252 first detection", 2),
        ("Parse internal IDs but validate by prefix", "454 parseable / 89 direct controls", 3),
        ("Treat IDs as ingestion-time stamps", "ID 2 s vs created_at 111 s median; late records drift together", 4),
        ("Use created_at only as a floor", "Maximum lags reach 349.538 h and 17,572.145 h", 5),
        ("Apply evidence-tiered anchors", "230 phase / 189 provisional / 381 dossier", 6),
        ("Keep EP pending and reject GRB IDs for phase", "EP causality pending; one GRB failed by 287.349 h", 7),
    ],
    columns=["decision", "evidence", "section"],
)
print(decisions.to_string(index=False))


                                    decision                                                       evidence  section
                 Reconstruct missing anchors                              672 lack t0; 296 are data-bearing        1
        Separate trigger and first detection                              548 trigger / 252 first detection        2
   Parse internal IDs but validate by prefix                             454 parseable / 89 direct controls        3
          Treat IDs as ingestion-time stamps ID 2 s vs created_at 111 s median; late records drift together        4
              Use created_at only as a floor                  Maximum lags reach 349.538 h and 17,572.145 h        5
               Apply evidence-tiered anchors                      230 phase / 189 provisional / 381 dossier        6
Keep EP pending and reject GRB IDs for phase              EP causality pending; one GRB failed by 287.349 h        7


**DECISION.** The augmented source index records the measured anchor, its
provenance and uncertainty, and whether it is admissible for phase matching.
The unresolved tiers stay explicit rather than being promoted without external
evidence.
